# Part 1 — Reinforcement Learning overview

This notebook gives the big RL map before individual algorithms. RL is learning by interaction: an agent observes state, takes action, receives reward, and improves a policy.

**Learning style:** mechanisms first → frameworks second → real systems third. The notebook is intentionally slow, explicit, and beginner-friendly.

In [ ]:
# Setup: run this first.
# Works from the repository root. In Colab, clone the repo first, then run from inside it.
from pathlib import Path
import sys, math, random
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    print('Tip: run this notebook from the repository root, or clone the repo in Colab first.')
sys.path.insert(0, str(ROOT))
print('Working directory:', ROOT)

## 1. Mental model

This notebook gives the big RL map before individual algorithms. RL is learning by interaction: an agent observes state, takes action, receives reward, and improves a policy.

Before code, write one sentence in your own words: *what problem does this topic solve?*

## 2. Mechanism and math

The Markov Decision Process is the base object:
\[
(S, A, P, R, \gamma)
\]
State `s`, action `a`, transition probability `P(s'|s,a)`, reward `R`, and discount `gamma`. The core quantity is return:
\[
G_t = r_t + \gamma r_{t+1}+\gamma^2 r_{t+2}+...\]
Value functions estimate expected return; policies choose actions.

## 3. From-scratch lab

Run a tiny hand-made environment so the interaction loop is visible before using Gymnasium or deep RL.

Read every line. The code avoids clever abstractions so you can see the mechanism.

In [ ]:
# A two-state MDP: start at 0. Action 1 goes to goal with reward; action 0 wastes time.
state = 0
Q = [[0.0, 0.0], [0.0, 0.0]]
alpha, gamma, eps = 0.3, 0.9, 0.2
import random

for episode in range(200):
    state = 0
    for t in range(5):
        action = random.randrange(2) if random.random() < eps else max(range(2), key=lambda a: Q[state][a])
        if state == 0 and action == 1:
            next_state, reward, done = 1, 1.0, True
        else:
            next_state, reward, done = state, 0.0, False
        target = reward + gamma * max(Q[next_state]) * (not done)
        Q[state][action] += alpha * (target - Q[state][action])
        state = next_state
        if done: break
print('Q table:', Q)
print('best action at start:', max(range(2), key=lambda a: Q[0][a]))

## 3.1 Code reading guide

When you read the previous cell, do not treat it as a black box. Trace it in this order:

1. **Inputs:** what are the given numbers, observations, states, rewards, or measurements?
2. **Internal variables:** what does each variable represent physically or mathematically?
3. **Update rule:** which line is the core mechanism from the math section?
4. **Output:** what should change if the mechanism is working?
5. **Failure case:** what parameter could make the example unstable, wrong, or unsafe?

This habit is the bridge between toy examples and real robotics code: every simulator, ROS node, policy, controller, or perception model still has inputs, state, an update rule, and outputs.

## 4. Framework/practice view

Frameworks supply environments, vectorization, logging, and tuned algorithms. Gymnasium standardizes `reset` and `step`; Stable-Baselines3 gives production-ready PPO/SAC/DQN; RLlib scales to distributed training.

The goal is not to replace understanding with APIs. The goal is to recognize the same mechanism when a library hides the details.

In [ ]:
try:
    import gymnasium as gym
    env = gym.make('CartPole-v1')
    obs, info = env.reset(seed=0)
    action = env.action_space.sample()
    next_obs, reward, terminated, truncated, info = env.step(action)
    print('obs shape:', obs.shape, 'sample action:', action, 'reward:', reward)
    env.close()
except ModuleNotFoundError as e:
    print('Install gymnasium to run this:', e)

## 4.1 Framework comparison checklist

After running or reading the framework cell, write a small mapping table for yourself:

| Question | Your answer |
|---|---|
| What object/function in the framework replaces the scratch code? |  |
| Which parameters match the math symbols? |  |
| What details does the framework hide? |  |
| What new engineering concerns appear? | installation, devices, logging, data formats, batching, safety, versioning |

This is where top-down learning becomes useful: you learn the professional API **without losing the mechanism**.

## 5. Real-system connection

Real systems use RL when actions affect future data: robot locomotion, drone aggressive flight, simulator-trained autonomous-driving policies, or RL fine-tuning after imitation learning. Always ask: what is the state, action, reward, reset condition, and safety constraint?

## 6. Exercises

1. For a drone landing task, define state/action/reward/done.
2. For a robot arm reaching task, define a bad reward and a better reward.
3. Explain why offline logs alone are imitation/offline RL, not normal online RL.

**Notebook habit:** after each exercise, add a short note explaining what changed and why it matters in a robot/car/drone/VLA stack.